# basic ML pipeline for weather forecasting
## notes
### Wind
speed (m/s)
deg -> direction, where 0\deg is due north
gust -> max gust

In [1]:
import requests
import os
import polars as pl
# import json
# from IPython.display import Image, display
# from io import BytesIO
from dotenv import load_dotenv

load_dotenv("../.env.local")

OWM_API_KEY = os.environ.get("OPENWEATHERMAP_API")

In [ ]:
def free_5d_forecast_from_zip(city):
    weather_url = f"https://api.openweathermap.org/data/2.5/forecast?q={city}&units=metric&APPID={OWM_API_KEY}"
    result = requests.get(weather_url)
    if result.status_code != 200:
        print(f'Access for {city} failed with {result.status_code}')
        print(result.text)
    return result.json()

    
def OWM_coord_forecast(coords: dict, units: str = "metric", exclude: list = []) -> dict:
    base_url = "https://api.openweathermap.org/data/3.0/onecall"
    required_parameters = f"appid={OWM_API_KEY}&lat={coords['lat']}&lon={coords['lon']}"

    exclude_result = "" if len(exclude) == 0 else f"?exclude={",".join(exclude)}"
    units_result = f"?units={units}"

    built_url = f"{base_url}?{required_parameters}" + units_result + exclude_result
    print(f"Calling: {built_url}")

    # res = requests.get(built_url)
    return built_url  # res.json()


def get_weather_icon(icon_id, scale: int = 1):
    # img = Image(icon.content)
    # display(img)
    url = f"https://openweathermap.org/img/wn/{icon_id}@2x.png"
    icon = requests.get(url)
    return icon

In [42]:
def geocode_from_zip(zip):
    location = f"{zip},US"
    geocode_url = f"https://api.openweathermap.org/geo/1.0/zip?zip={location}&APPID={OWM_API_KEY}"
    result = requests.get(geocode_url)
    if result.status_code != 200:
        print(f'Access for {zip} failed with {result.status_code}')
        print(result.text)
    return result.json()


def forecast_5d_from_coord(lat, lon):
    weather_url = f"https://api.openweathermap.org/data/2.5/forecast?appid={OWM_API_KEY}&lat={lat}&lon={lon}&units=metric"
    result = requests.get(weather_url)
    if result.status_code != 200:
        print(f'Access for lat: {lat}, lon: {lon} failed with {result.status_code}')
        print(result.text)
    return result.json()


def reshape_city_forecast(forecast):
    fc_df = pl.LazyFrame(forecast["list"])
    fc_df_cols = fc_df.collect_schema().names()
    fc_df_time = fc_df.select(
        "dt_txt",
        pl.from_epoch(pl.col("dt"), time_unit="s").sort()
    )
    location_df = pl.LazyFrame(forecast["city"]).select(pl.all(), pl.col("coord").struct.unnest()).drop("coord").select(
        pl.col("id").alias("city_id"),
        pl.col("name").alias("city_name"),
        "population",
        "lat",
        "lon"
    )
    x_time_df = fc_df_time.join(location_df, how="cross")

    if "snow" in fc_df_cols:
        snow_expr = pl.col("snow").struct.unnest().name.suffix("_snow_mm").fill_null(0)
    else:
        snow_expr = pl.lit(0.0).alias("3h_snow_mm")

    if "rain" in fc_df_cols:
        rain_expr = pl.col("rain").struct.unnest().name.suffix("_rain_mm").fill_null(0)
    else:
        rain_expr = pl.lit(0.0).alias("3h_rain_mm")

    return fc_df.select(
        pl.from_epoch(pl.col("dt"), time_unit="s").set_sorted(),
        pl.col("dt_txt"),
        pl.col("sys").struct.field("pod").name.replace("pod", "part_of_day").cast(pl.Categorical),
        pl.col("main").struct.unnest(),
        pl.col("wind").name.prefix_fields("wind_").struct.unnest(),
        pl.col("visibility").alias("vis_meters"),
        pl.col("clouds").struct.unnest().name.replace("all", "cloud_cover_pct"),
        pl.col("pop").alias("p_of_precipitation").cast(pl.Float64),
        snow_expr,
        rain_expr,
        pl.col("weather").list.explode().struct.field("description").alias("weather_desc").cast(pl.Categorical)
    ).drop(["temp_kf", "pressure"]).join(x_time_df, how="inner", on="dt")

In [43]:
list_of_cities = ["02472", "01907", "01945", "01940", "60606"]
# forecasts = free_5d_forecast_from_zip(list_of_cities[0])

coord_list = [{"lat": res["lat"], "lon": res["lon"]} for res in [geocode_from_zip(zip) for zip in list_of_cities]]

forecasts: pl.LazyFrame = [reshape_city_forecast(forecast_5d_from_coord(coord["lat"], coord["lon"])) for coord in coord_list]

result_frame = pl.LazyFrame(schema=forecasts[0].collect_schema())

for fc in forecasts:
    result_frame = pl.concat([result_frame, fc])

weather_forecasts_df = result_frame.drop("dt_txt_right").collect()

In [44]:
weather_forecasts_df.columns

['dt',
 'dt_txt',
 'part_of_day',
 'temp',
 'feels_like',
 'temp_min',
 'temp_max',
 'sea_level',
 'grnd_level',
 'humidity',
 'wind_speed',
 'wind_deg',
 'wind_gust',
 'vis_meters',
 'cloud_cover_pct',
 'p_of_precipitation',
 '3h_snow_mm',
 '3h_rain_mm',
 'weather_desc',
 'city_id',
 'city_name',
 'population',
 'lat',
 'lon']

In [53]:
chart = (
    weather_forecasts_df.plot.bar(
        x="dt", 
        y="3h_snow_mm", 
        color="city_name"
    ).properties(width=500, title="snow")
    .configure_scale(zero=False)
    .configure_axisX(tickMinStep=1),
    weather_forecasts_df.plot.bar(
        x="dt", 
        y="3h_rain_mm", 
        color="city_name"
    ).properties(width=500, title="rain")
    .configure_scale(zero=False)
    .configure_axisX(tickMinStep=1)
)

chart

(alt.Chart(...), alt.Chart(...))